# Multi-Stage Progress Example

This notebook demonstrates complex multi-stage tasks with nested progress bars and different throttling strategies.

In [1]:
from fasthtml.common import *
import uuid, time, json
import random

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_sizes, badge_styles, badge_colors
from cjm_fasthtml_daisyui.components.data_input.range_slider import range_dui, range_colors
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.layout import display_tw
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_align
from cjm_fasthtml_tailwind.utilities.sizing import w, h, container
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_wrap, gap
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.CYBERPUNK)

# Initialize
monitor = ProgressMonitor(keep_history=True)
runner = JobRunner(monitor)

In [4]:
# Complex multi-stage pipeline
def data_pipeline(dataset_size=1000):
    from tqdm import tqdm
    import time
    
    results = {
        "stages": [],
        "metrics": {}
    }
    
    # Stage 1: Data Collection
    print("Starting Stage 1: Data Collection")
    collected_data = []
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source A"):
        time.sleep(0.002)
        collected_data.append(f"source_a_{i}")
    
    for i in tqdm(range(dataset_size // 4), desc="1. Collecting from Source B"):
        time.sleep(0.002)
        collected_data.append(f"source_b_{i}")
    
    results["stages"].append("Data Collection Complete")
    results["metrics"]["collected"] = len(collected_data)
    
    # Stage 2: Data Validation
    print("Starting Stage 2: Data Validation")
    valid_data = []
    for item in tqdm(collected_data, desc="2. Validating Data"):
        time.sleep(0.001)
        if random.random() > 0.1:  # 90% pass validation
            valid_data.append(item)
    
    results["stages"].append("Data Validation Complete")
    results["metrics"]["validated"] = len(valid_data)
    
    # Stage 3: Data Transformation
    print("Starting Stage 3: Data Transformation")
    
    # Sub-stage 3.1: Normalization
    normalized = []
    for item in tqdm(valid_data[:len(valid_data)//2], desc="3.1 Normalizing"):
        time.sleep(0.002)
        normalized.append(f"norm_{item}")
    
    # Sub-stage 3.2: Encoding
    encoded = []
    for item in tqdm(valid_data[len(valid_data)//2:], desc="3.2 Encoding"):
        time.sleep(0.002)
        encoded.append(f"enc_{item}")
    
    # Sub-stage 3.3: Merging
    merged = []
    for i in tqdm(range(min(len(normalized), len(encoded))), desc="3.3 Merging"):
        time.sleep(0.001)
        merged.append((normalized[i], encoded[i]))
    
    results["stages"].append("Data Transformation Complete")
    results["metrics"]["transformed"] = len(merged)
    
    # Stage 4: Analysis
    print("Starting Stage 4: Analysis")
    
    # Parallel analysis tasks
    for i in tqdm(range(100), desc="4.1 Statistical Analysis"):
        time.sleep(0.01)
    
    for i in tqdm(range(80), desc="4.2 Pattern Detection"):
        time.sleep(0.012)
    
    for i in tqdm(range(60), desc="4.3 Anomaly Detection"):
        time.sleep(0.015)
    
    results["stages"].append("Analysis Complete")
    results["metrics"]["patterns_found"] = random.randint(10, 50)
    results["metrics"]["anomalies"] = random.randint(1, 10)
    
    # Stage 5: Report Generation
    print("Starting Stage 5: Report Generation")
    
    for i in tqdm(range(50), desc="5.1 Generating Charts"):
        time.sleep(0.02)
    
    for i in tqdm(range(30), desc="5.2 Writing Summary"):
        time.sleep(0.03)
    
    for i in tqdm(range(20), desc="5.3 Finalizing Report"):
        time.sleep(0.04)
    
    results["stages"].append("Report Generation Complete")
    results["report"] = "pipeline_report.pdf"
    
    return results

In [5]:
# Nested loop example
def nested_processing():
    from tqdm import tqdm
    import time
    
    results = []
    
    # Outer loop - batches
    batches = 5
    items_per_batch = 20
    
    for batch in tqdm(range(batches), desc="Processing Batches"):
        batch_results = []
        
        # Inner loop - items in batch
        for item in tqdm(range(items_per_batch), desc=f"Batch {batch+1} Items", leave=False):
            time.sleep(0.05)
            batch_results.append(f"batch_{batch}_item_{item}")
        
        results.append(batch_results)
        time.sleep(0.1)  # Pause between batches
    
    return results

In [6]:
# UI for multi-stage progress
from cjm_fasthtml_daisyui.core.testing import create_test_page

@rt("/")
def index():
    return create_test_page(
        "Multi-Stage Progress Demo",
        Div(
            H1("Multi-Stage Pipeline Progress", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Pipeline controls
            Div(
                H2("Pipeline Configuration", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Label("Dataset Size:", cls=str(label)),
                    Input(
                        id="dataset-size",
                        name="dataset_size",
                        type="range",
                        min="100",
                        max="2000",
                        value="1000",
                        cls=combine_classes(range_dui, range_colors.primary),
                        oninput="document.getElementById('size-display').textContent = this.value"
                    ),
                    Span("1000", id="size-display", cls=combine_classes(badge, badge_sizes.lg, m.l(2))),
                    cls=combine_classes(m.b(4))
                ),
                Div(
                    Form(
                        Input(type="hidden", name="task_type", value="pipeline"),
                        Input(type="hidden", name="dataset_size", id="dataset-size-hidden", value="1000"),
                        Button(
                            "Start Pipeline", 
                            type="submit",
                            id="start-pipeline-btn",
                            cls=combine_classes(btn, btn_colors.primary),
                            onclick="document.getElementById('dataset-size-hidden').value = document.getElementById('dataset-size').value"
                        ),
                        hx_post="/start_task",
                        hx_target="#progress-container",
                        hx_swap="innerHTML",
                        cls=combine_classes(display_tw.inline_block, m.r(0.5))
                    ),
                    Form(
                        Input(type="hidden", name="task_type", value="nested"),
                        Button(
                            "Run Nested Task",
                            type="submit",
                            id="run-nested-btn",
                            cls=combine_classes(btn, btn_colors.secondary)
                        ),
                        hx_post="/start_task",
                        hx_target="#progress-container",
                        hx_swap="innerHTML",
                        cls=combine_classes(display_tw.inline_block)
                    ),
                    id="task-buttons"  # Add ID to buttons container for updating
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress container
            Div(id="progress-container", cls=str(m.b(6))),
            
            # Results
            Div(
                H2("Results", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Pre("No results yet", id="results", cls=combine_classes(bg_dui.base_300, p(4), rounded())),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [7]:
# Storage for results
job_results = {}

def check_has_active_jobs():
    """Check if any jobs are currently running"""
    jobs = monitor.all()
    return any(not job['completed'] for job in jobs.values())

def render_task_buttons(disabled=None):
    """Render task buttons with appropriate state"""
    # If disabled is not explicitly passed, check monitor for active jobs
    if disabled is None:
        disabled = check_has_active_jobs()
    
    btn_classes_pipeline = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    btn_classes_nested = combine_classes(
        btn, 
        btn_colors.secondary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Div(
        Form(
            Input(type="hidden", name="task_type", value="pipeline"),
            Input(type="hidden", name="dataset_size", id="dataset-size-hidden", value="1000"),
            Button(
                "Start Pipeline", 
                type="submit",
                id="start-pipeline-btn",
                cls=btn_classes_pipeline,
                disabled=disabled,
                onclick="document.getElementById('dataset-size-hidden').value = document.getElementById('dataset-size').value" if not disabled else None
            ),
            hx_post="/start_task" if not disabled else None,
            hx_target="#progress-container",
            hx_swap="innerHTML",
            cls=combine_classes(display_tw.inline_block, m.r(0.5))
        ),
        Form(
            Input(type="hidden", name="task_type", value="nested"),
            Button(
                "Run Nested Task",
                type="submit",
                id="run-nested-btn",
                cls=btn_classes_nested,
                disabled=disabled
            ),
            hx_post="/start_task" if not disabled else None,
            hx_target="#progress-container",
            hx_swap="innerHTML",
            cls=combine_classes(display_tw.inline_block)
        ),
        id="task-buttons",
        hx_swap_oob="true"  # This enables out-of-band swap
    )

# API endpoints using FastHTML conventions
@rt("/start_task")
async def start_task(request):
    form = await request.form()
    task_type = form.get('task_type', 'pipeline')
    dataset_size = int(form.get('dataset_size', 1000))
    
    job_id = str(uuid.uuid4())
    
    if task_type == 'nested':
        task_name = "Nested Processing"
        def wrapper():
            result = nested_processing()
            job_results[job_id] = {"nested_results": result, "total_items": sum(len(b) for b in result)}
        
        runner.start(
            job_id,
            wrapper,
            patch_kwargs={
                "min_delta_pct": 5,
                "min_update_interval": 0.1,
                "emit_initial": True
            }
        )
    else:
        task_name = "Data Pipeline"
        def wrapper():
            result = data_pipeline(dataset_size)
            job_results[job_id] = result
        
        runner.start(
            job_id,
            wrapper,
            patch_kwargs={
                "min_delta_pct": 1,
                "min_update_interval": 0.05,
                "emit_initial": True
            }
        )
    
    # Return initial progress UI with disabled buttons
    return Div(
        # Include disabled buttons via out-of-band swap
        render_task_buttons(disabled=True),
        # Progress display
        Div(
            H2("Pipeline Progress", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
            Div(
                _render_full_progress(job_id, None, task_type),
                hx_get=f"/job_progress?job_id={job_id}&task_type={task_type}",
                hx_trigger="load",
                hx_swap="innerHTML",
                id=f"progress-content-{job_id}"
            ),
            cls=combine_classes(card, bg_dui.base_200, p(6))
        )
    )

def _render_full_progress(job_id, snapshot, task_type):
    """Render full progress including stage indicators"""
    stages = ['Collection', 'Validation', 'Transformation', 'Analysis', 'Report'] if task_type == 'pipeline' else []
    
    # Detect active stages if we have a snapshot
    stage_statuses = {}
    if snapshot and stages:
        stage_statuses = _detect_active_stages(snapshot['bars'])
    
    # Build stage indicators
    stage_indicators = []
    if stages:
        for stage in stages:
            status = stage_statuses.get(stage, 'pending')
            if status == 'complete':
                badge_class = combine_classes(badge, badge_sizes.lg, badge_colors.success)
            elif status == 'active':
                badge_class = combine_classes(badge, badge_sizes.lg, badge_colors.warning)
            else:
                badge_class = combine_classes(badge, badge_sizes.lg, badge_styles.outline)
            
            stage_indicators.append(
                Span(stage, cls=badge_class)
            )
    
    # Build progress content
    if not snapshot:
        progress_content = Div(
            P("Overall Progress:", cls=str(font_weight.bold)),
            Progress(value="0", max="100", cls=combine_classes(progress, progress_colors.primary, w.full, h(6))),
            P("0%", cls=combine_classes(text_align.center, m.t(1))),
            Div(
                H3("Starting...", cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
                Div(cls=str(space.y(3)))
            )
        )
    else:
        # Build progress bars
        bars_html = []
        for bar_id, bar in snapshot['bars'].items():
            bar_color = _get_bar_color(bar.description)
            bars_html.append(
                Div(
                    P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                      cls=combine_classes(font_size.sm, font_weight.semibold, m.b(1))),
                    Progress(
                        value=str(bar.progress), 
                        max="100", 
                        cls=combine_classes(progress, bar_color, w.full)
                    ),
                    cls=combine_classes(p(3), bg_dui.base_300, rounded())
                )
            )
        
        progress_content = Div(
            P("Overall Progress:", cls=str(font_weight.bold)),
            Progress(
                value=str(snapshot['overall_progress']), 
                max="100", 
                cls=combine_classes(progress, progress_colors.primary, w.full, h(6))
            ),
            P(f"{snapshot['overall_progress']:.1f}%", cls=combine_classes(text_align.center, m.t(1))),
            Div(
                H3("Active Tasks:", cls=combine_classes(font_weight.semibold, m.b(3), m.t(6))),
                Div(*bars_html, cls=str(space.y(3)))
            )
        )
    
    # Combine everything
    return Div(
        # Stage indicators
        Div(
            *stage_indicators,
            cls=combine_classes(flex_display, flex_wrap.wrap, gap(2), m.b(4))
        ) if stage_indicators else None,
        # Progress content
        progress_content
    )

def _detect_active_stages(bars):
    """Detect active and completed stages from progress bars"""
    stages = {
        'Collection': 'pending',
        'Validation': 'pending', 
        'Transformation': 'pending',
        'Analysis': 'pending',
        'Report': 'pending'
    }
    
    # Track which stages we've seen
    seen_stages = set()
    
    for bar_id, bar in bars.items():
        desc = bar.description or ""
        progress = bar.progress
        
        if '1.' in desc or 'Collecting' in desc:
            seen_stages.add('Collection')
            stages['Collection'] = 'complete' if progress >= 100 else 'active'
        elif '2.' in desc or 'Validating' in desc:
            seen_stages.add('Validation')
            stages['Validation'] = 'complete' if progress >= 100 else 'active'
            # Mark previous stage as complete
            if 'Collection' not in seen_stages:
                stages['Collection'] = 'complete'
        elif '3.' in desc or 'Transform' in desc or 'Normaliz' in desc or 'Encod' in desc or 'Merg' in desc:
            seen_stages.add('Transformation')
            stages['Transformation'] = 'complete' if progress >= 100 else 'active'
            # Mark previous stages as complete
            if 'Validation' not in seen_stages:
                stages['Validation'] = 'complete'
            if 'Collection' not in seen_stages:
                stages['Collection'] = 'complete'
        elif '4.' in desc or 'Analysis' in desc or 'Pattern' in desc or 'Anomaly' in desc:
            seen_stages.add('Analysis')
            stages['Analysis'] = 'complete' if progress >= 100 else 'active'
            # Mark previous stages as complete
            for prev_stage in ['Collection', 'Validation', 'Transformation']:
                if prev_stage not in seen_stages:
                    stages[prev_stage] = 'complete'
        elif '5.' in desc or 'Report' in desc or 'Chart' in desc or 'Summary' in desc:
            seen_stages.add('Report')
            stages['Report'] = 'complete' if progress >= 100 else 'active'
            # Mark previous stages as complete
            for prev_stage in ['Collection', 'Validation', 'Transformation', 'Analysis']:
                if prev_stage not in seen_stages:
                    stages[prev_stage] = 'complete'
    
    return stages

def _get_bar_color(description):
    """Get progress bar color based on description"""
    if not description:
        return progress_colors.accent
    
    if '1.' in description:
        return progress_colors.info
    elif '2.' in description:
        return progress_colors.success
    elif '3.' in description:
        return progress_colors.warning
    elif '4.' in description:
        return progress_colors.error
    elif '5.' in description:
        return progress_colors.primary
    else:
        return progress_colors.accent

In [8]:
@rt("/job_progress")
def job_progress(job_id: str, task_type: str = 'pipeline'):
    """Returns current progress state for a job"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        # Job not started yet - continue polling until it starts
        return Div(
            _render_full_progress(job_id, None, task_type),
            hx_get=f"/job_progress?job_id={job_id}&task_type={task_type}",
            hx_trigger="load delay:100ms",  # Keep polling until task starts
            hx_swap="outerHTML"
        )
    
    # Check if completed
    if snapshot['completed']:
        results = job_results.get(job_id, {})
        
        # Return final progress display with re-enabled buttons via out-of-band swap
        return Div(
            # Re-enable buttons via out-of-band swap (using None to auto-check monitor)
            render_task_buttons(disabled=None),
            # Final progress display
            _render_full_progress(job_id, snapshot, task_type),
            Script(f"""
                // Update results
                document.getElementById('results').textContent = {json.dumps(json.dumps(results, indent=2))};
            """)
            # No hx_trigger here - polling stops
        )
    
    # Task is in progress - continue polling
    return Div(
        _render_full_progress(job_id, snapshot, task_type),
        hx_get=f"/job_progress?job_id={job_id}&task_type={task_type}",
        hx_trigger="load delay:100ms",  # Continue polling while in progress
        hx_swap="outerHTML"
    )

In [9]:
# Optional SSE endpoint for direct streaming (not used in HTMX polling approach)
@rt("/stream")
def stream(job_id: str):
    """SSE endpoint - available for direct EventSource usage if needed"""
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.05,
            heartbeat=15.0,
            wait_for_start=True,
            start_timeout=10.0
        )
    )

In [10]:
@rt("/results")
def get_results(job_id: str):
    """Get results for a completed job"""
    if job_id in job_results:
        return Pre(json.dumps(job_results[job_id], indent=2), cls=combine_classes(bg_dui.base_300, p(4), rounded()))
    return Div("Results not available", cls=combine_classes(badge, badge_colors.error))

In [11]:
# Start server for Jupyter
from cjm_fasthtml_daisyui.core.testing import start_test_server
from fasthtml.jupyter import HTMX
from IPython.display import display

server = start_test_server(app)
display(HTMX())

In [12]:
# Stop server when done
server.stop()